### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 4_smoothing_and_LOESS_filter
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es) & Bàrbara Barceló-Llull (bbarcelo@imedea.uib-csic.es)
### Data from BioSWOT experiment
# 
**DESCRIPTION**:
 This script applies a differentiated smoothing technique (Split Smoothing) 
 to the lag-corrected data generated in Step 3. 
 - Temperature: Undergoes a gentle 5-point rolling median filter to remove 
   residual electrical spikes while preserving fine-scale physical structures.
 - Salinity: Undergoes both the median filter and a physical LOESS 
   (Locally Estimated Scatterplot Smoothing) filter adjusted to a 10m 
   vertical scale. This effectively dampens the high-frequency noise inherent 
   to thermal mass and conductivity cell mismatches.
#
 INPUT: Lag-corrected NetCDF files (*_step3_lag.nc).
#
 OUTPUT: Smoothed NetCDF files (*_step4_loess.nc).
### ==============================================================================

In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm
import gsw
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION BIOSWOT
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_bioswot_nc_final_processing")

# Input: Folder generated in Step 3 of BioSWOT
DIR_IN = BASE_ROOT / "Data" / "processed_step3_lag"
DIR_OUT = BASE_ROOT / "Data" / "processed_step4_loess"
DIR_OUT.mkdir(parents=True, exist_ok=True)

# Scientific Parameters
MEDIAN_WINDOW = 5              # Window in scans
VERTICAL_SCALE_SALT = 10.0     # LOESS Scale in meters

# ==========================================
# 2. FUNCTIONS
# ==========================================

def apply_loess(data, depth, scale_meters):
    """Apply LOESS with dynamic adjustment of the frac parameter."""
    mask = np.isfinite(data) & np.isfinite(depth)
    if mask.sum() < 20: return data 
    
    y, x = data[mask], depth[mask]
    z_range = x.max() - x.min()
    
    # Calculate Frac: proportion of points for the scale_meters window
    if z_range < scale_meters: 
        frac = 0.8 
    else:
        frac = scale_meters / z_range
        frac = np.clip(frac, 0.02, 0.5) 
        
    smoothed = sm.nonparametric.lowess(y, x, frac=frac, it=0, return_sorted=False)
    
    out = np.full_like(data, np.nan)
    out[mask] = smoothed
    return out

# ==========================================
# 3. PROCESSING
# ==========================================
print(f"Initializing Step 4 BioSWOT (S_scale: {VERTICAL_SCALE_SALT}m)...")

files = sorted(list(DIR_IN.glob("*_step3_lag.nc")))

if not files:
    print(f"❌ No files found in {DIR_IN}. Please check Step 3.")
else:
    for f in files:
        try:
            with xr.open_dataset(f) as ds:
                p = ds.pressure.values
                
                # Load variables from Step 3 BioSWOT
                # temp = t1_lagged | salt = salinity_lagged
                t_lag = ds['t1_lagged'].values if 't1_lagged' in ds else ds['t1'].values
                s_lag = ds['salinity_lagged'].values if 'salinity_lagged' in ds else np.nan
                
                if np.all(np.isnan(s_lag)):
                    print(f"⚠️ Skipping {f.name}: No valid salinity data."); continue

                # --- 2. SMOOTHING TEMPERATURE ---
                t_smooth = pd.Series(t_lag).rolling(window=MEDIAN_WINDOW, center=True, min_periods=1).median().values
                
                # --- 3. SMOOTHING SALINITY ---
                # A) Previous median for spikes
                s_median = pd.Series(s_lag).rolling(window=MEDIAN_WINDOW, center=True, min_periods=1).median().values
                # B) LOESS
                s_smooth = apply_loess(s_median, p, scale_meters=VERTICAL_SCALE_SALT)
                
                # --- 4. SAVE ---
                ds_out = ds.copy(deep=True)
                ds_out['t1_smooth'] = (('scan',), t_smooth)
                ds_out['salinity_smooth'] = (('scan',), s_smooth)
                
                # Metadata
                ds_out['t1_smooth'].attrs = {'units': 'degC', 'comment': 'Median filter (5 pts) applied to t1_lagged'}
                ds_out['salinity_smooth'].attrs = {'units': 'PSU', 'comment': f'LOESS smoothing ({VERTICAL_SCALE_SALT}m) applied to salinity_lagged'}
                
                out_name = f.name.replace("_step3_lag.nc", "_step4_loess.nc")
                ds_out.to_netcdf(DIR_OUT / out_name)
                
            print(f"   ✅ {f.name} -> {out_name}")
            
        except Exception as e:
            print(f"❌ Error in {f.name}: {e}")

print(f"\n Done! Data of BioSWOT smoothed in {DIR_OUT}")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Initializing Step 4 BioSWOT (S_scale: 10.0m)...
   ✅ PL10_mvp_2023-04-29_224406_step3_lag.nc -> PL10_mvp_2023-04-29_224406_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_225254_step3_lag.nc -> PL10_mvp_2023-04-29_225254_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_230112_step3_lag.nc -> PL10_mvp_2023-04-29_230112_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_230930_step3_lag.nc -> PL10_mvp_2023-04-29_230930_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_231748_step3_lag.nc -> PL10_mvp_2023-04-29_231748_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_232609_step3_lag.nc -> PL10_mvp_2023-04-29_232609_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_233428_step3_lag.nc -> PL10_mvp_2023-04-29_233428_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_234249_step3_lag.nc -> PL10_mvp_2023-04-29_234249_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_235113_step3_lag.nc -> PL10_mvp_2023-04-29_235113_step4_loess.nc
   ✅ PL10_mvp_2023-04-29_235937_step3_lag.nc -> PL10_mvp_2023-04-29_235937_step4_loess.nc
   ✅ PL10_mvp_2023-04-30_000800_step3_lag.nc -> PL10